Imports the `pyspark` package so its APIs (and `SparkSession` later on) are available in this notebook.

In [1]:
import pyspark

`pyspark.__version__` reports the installed PySpark version, used here to sanity-check that the `uv`-managed environment has the expected version.

In [4]:
pyspark.__version__

'4.2.0'

`pyspark.__file__` shows the on-disk path of the installed package, useful for confirming which environment's PySpark is actually being imported.

In [2]:
pyspark.__file__


'/home/harsh/data_engineering_zoomcamp/Data-engineering-zoomcamp/batch_processing_spark/.venv/lib/python3.13/site-packages/pyspark/__init__.py'

Imports `SparkSession`, the entry point for creating and configuring a Spark application — needed before any DataFrame or SQL operations can run.

In [3]:
from pyspark.sql import SparkSession

`SparkSession.builder...getOrCreate()` creates (or reuses) a Spark session running locally with `local[*]`, using all available CPU cores as the "cluster" — appropriate for local experimentation instead of a real Spark cluster.

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/11 09:31:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Downloads the NYC TLC taxi zone lookup CSV so there's a real dataset on disk to read into a Spark DataFrame in the next steps.

In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-08-11 09:34:42--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.111, 3.170.186.229, 3.170.186.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-08-11 09:34:42 (43.4 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



Previews the raw CSV's first few lines to check its structure (headers, columns) before letting Spark infer/parse it.

In [7]:
!head taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


`spark.read.option("header", "true").csv(...)` reads the CSV into a Spark DataFrame, telling Spark to treat the first row as column names rather than data.

In [11]:
 df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

`df.show()` triggers Spark's lazy evaluation and prints the first 20 rows, used here to visually confirm the CSV loaded correctly.

In [12]:
df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

`df.write.parquet('zones')` writes the DataFrame out to the columnar Parquet format, demonstrating Spark's standard output format for downstream processing (more compact and faster to read than CSV).

In [13]:
df.write.parquet('zones')

Lists the working directory's contents to confirm the `zones` Parquet output was actually written to disk.

In [14]:
!ls -lh

total 224K
-rw-rw-r-- 1 harsh harsh    0 Aug 11 09:01 README.md
-rw-rw-r-- 1 harsh harsh 4.8K Aug 11 09:35 Untitled.ipynb
-rw-rw-r-- 1 harsh harsh  100 Aug 11 09:01 main.py
-rw-rw-r-- 1 harsh harsh  236 Aug 11 09:19 pyproject.toml
-rw-rw-r-- 1 harsh harsh  13K Feb 22  2024 taxi_zone_lookup.csv
-rw-rw-r-- 1 harsh harsh  239 Aug 11 09:06 test_script.py
-rw-rw-r-- 1 harsh harsh 182K Aug 11 09:19 uv.lock
drwxr-xr-x 2 harsh harsh 4.0K Aug 11 09:36 zones


Note-to-self reminder that Spark exposes a web UI on port 4040 (while the session is active) for inspecting completed and running jobs, stages, and tasks.

In [15]:
# open port 4040 to look at spark jobs which have been completed